# 10k node stress test

Spawns **10,000 nodes** (500 chains, 20 nodes deep) and times every operation.
Nodes are metadata-only so the timings are dominated by ancestree itself, not artifact I/O.
Re-run top to bottom; the store is wiped at the start of each run.

In [1]:
import shutil, time, random, io, contextlib
from pathlib import Path
import ancestree

ROOT = Path("benchmark_10k_store")
if ROOT.exists():
    shutil.rmtree(ROOT)

rules = {"ingest": [None], "transform": ["ingest", "transform"]}
store = ancestree.LineageStore(ROOT, rules=rules, gen_triggers=["ingest"])

In [2]:
import numpy as np

def timed(label, fn, reps=100):
    times, result = [], None
    for _ in range(reps):
        t = time.perf_counter()
        result = fn()
        times.append((time.perf_counter() - t) * 1000)
    hits = f"{len(result)} hits" if isinstance(result, list) else type(result).__name__
    # print(f"{label:<50} best of {reps}: {min(times):8.2f} ms   ({hits})")
    # print(f"{label:<50} worst of {reps}: {max(times):8.2f} ms   ({hits})")
    print(f"{label:<50} median of {reps}: {np.median(times):8.2f} ms   ({hits})")
    return result

## 1. Create 10,000 nodes

In [3]:
N = 1_000
CHAIN_LEN = 20

random.seed(0)
ids, parent = [], None
t0 = time.perf_counter()
for i in range(N):
    if i % CHAIN_LEN == 0:
        step, parent = "ingest", None
    else:
        step = "transform"
    with store.create_node(step_type=step, parent=parent) as node:
        node.add_meta("accuracy", round(random.random(), 4))
        node.add_meta("batch", i // CHAIN_LEN)
    parent = node
    ids.append(node.node_id)
    if (i + 1) % 1000 == 0:
        print(f"{i+1:>6} nodes   {time.perf_counter()-t0:6.1f}s elapsed")

elapsed = time.perf_counter() - t0
print(f"\ncreated {N} nodes in {elapsed:.1f}s  ({elapsed/N*1000:.2f} ms/node)")

  1000 nodes      1.2s elapsed

created 1000 nodes in 1.2s  (1.18 ms/node)


## 2. Searching and querying (warm, in-memory)

In [ ]:
_ = timed("find_node(step_type='ingest')", lambda: store.find_node(step_type="ingest"))
_ = timed("find_node(batch=250)", lambda: store.find_node(batch=20))
_ = timed("find_node(lambda accuracy > 0.99)", lambda: store.find_node(accuracy=lambda a: a is not None and a > 0.99))
_ = timed("get_most_recent_node(step_type='transform')", lambda: store.get_most_recent_node(step_type="transform"))
_ = timed("get_child_nodes(first root)", lambda: store.get_child_nodes(ids[0]))

find_node(step_type='ingest')                      median of 100:     2.80 ms   (50 hits)
find_node(batch=250)                               median of 100:     0.20 ms   (0 hits)
find_node(lambda accuracy > 0.99)                  median of 100:     0.48 ms   (5 hits)
get_most_recent_node(step_type='transform')        median of 100:     0.45 ms   (Node)
get_child_nodes(first root)                        median of 100:     0.31 ms   (1 hits)


## 3. Lineage traversal

In [21]:
deepest = ids[CHAIN_LEN - 1]  # last node of the first chain (depth 20)
_ = timed("get_lineage (depth 20)", lambda: store.get_lineage(deepest))
_ = timed("find_in_lineage(lambda accuracy > 0.5)", lambda: store.find_in_lineage(deepest, accuracy=lambda a: a is not None and a > 0.5))

KeyError: "Node '7068eb03' not found in the index. It may have been pruned without recursive=True. Call store.rebuild_db_from_disk() to resync the index."

## 4. Cold start (fresh process simulation: new store instance loads the snapshot)

In [22]:
t = time.perf_counter()
store2 = ancestree.LineageStore(ROOT)
first = store2.find_node(batch=0)
print(f"cold start (init + load index + first query):    {(time.perf_counter()-t)*1000:8.2f} ms")
_ = timed("same query again (warm)", lambda: store2.find_node(batch=0))

cold start (init + load index + first query):      317.68 ms
same query again (warm)                            median of 100:    11.59 ms   (0 hits)


## 5. Full index rebuild (the only disk crawl — recovery path)

In [6]:
t = time.perf_counter()
store.rebuild_db_from_disk()
print(f"rebuild_db_from_disk ({N} meta.json files): {time.perf_counter()-t:.2f} s")

rebuild_db_from_disk (50000 meta.json files): 3.20 s


## 6. Prune one chain (20 nodes)

In [7]:
t = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    store.prune(ids[0], dry_run=True)
print(f"prune dry-run (chain of {CHAIN_LEN}):  {(time.perf_counter()-t)*1000:8.2f} ms")

t = time.perf_counter()
store.prune(ids[0], recursive=True, dry_run=False)
print(f"prune for real (chain of {CHAIN_LEN}): {(time.perf_counter()-t)*1000:8.2f} ms")

prune dry-run (chain of 20):    233.80 ms
prune for real (chain of 20):  1448.01 ms


## 7. Web graph generation

Timing the HTML generation only — don't actually open a 10k node graph in a browser unless you enjoy fan noise.

In [8]:
t = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    store.generate_web_graph()
print(f"generate_web_graph: {time.perf_counter()-t:.2f} s")
html = ROOT / "interactive_pipeline.html"
print(f"HTML size: {html.stat().st_size/1e6:.1f} MB")

generate_web_graph: 6.87 s
HTML size: 42.5 MB


## 8. What's on disk

In [23]:
files = [f for f in ROOT.rglob("*") if f.is_file()]
snap = ROOT / ".index.json"
print(f"snapshot (.index.json): {snap.stat().st_size/1e6:.2f} MB")
print(f"store total:            {sum(f.stat().st_size for f in files)/1e6:.1f} MB across {len(files)} files")

snapshot (.index.json): 9.40 MB
store total:            94.9 MB across 49983 files


In [10]:
# Uncomment to clean up:
# shutil.rmtree(ROOT)